# [HumanEvalPack](https://huggingface.co/datasets/bigcode/humanevalpack)
HumanEvalPack is an extension of OpenAI's HumanEval to cover 6 total languages across 3 tasks. The Python split is exactly the same as OpenAI's Python HumanEval. The other splits are translated by humans (similar to HumanEval-X but with additional cleaning, see [here](https://github.com/bigcode-project/octopack/tree/main/evaluation/create/humaneval-x#modifications-muennighoff)). Refer to the OctoPack paper for more details.

Languages: Python, JavaScript, Java, Go, C++, Rust

The data fields are the same among all splits:

- `task_id`: Indicates the language (Python/JavaScript/Java/Go/C++/Rust) and task id (from 0 to 163) of the problem  
- `prompt`: the prompt for models relying on code continuation  
- `declaration`: the declaration of the function (same as prompt but without the docstring)  
- `canonical_solution`: the correct solution passing all unit tests for the problem  
- `buggy_solution`: same as canonical_solution but with a subtle human-written bug causing the unit tests to fail  
- `bug_type`: the type of the bug in buggy_solution (one of [missing logic, excess logic, value misuse, operator misuse, variable misuse, function misuse])  
- `failure_symptoms`: the problem the bug causes (one of [incorrect output, stackoverflow, infinite loop])  
- `entry_point`: the name of the function  
- `import`: imports necessary for the solution (only present for Go)  
- `test_setup`: imports necessary for the test execution (only present for Go)  
- `test`: the unit tests for the problem  
- `example_test`: additional unit tests different from test that could be e.g. provided to the model (these are not used in the paper)  
- `signature`: the signature of the function  
- `docstring`: the docstring describing the problem  
- `instruction`: an instruction for HumanEvalSynthesize in the form Write a {language_name} function {signature} to solve the following problem:\n{docstring}

In [1]:
# pip install -q datasets
from datasets import load_dataset
# Languages: "python", "js", "java", "go", "cpp", "rust"
ds = load_dataset("bigcode/humanevalpack", "cpp")
ds


/home/gkoren/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    test: Dataset({
        features: ['task_id', 'prompt', 'declaration', 'canonical_solution', 'buggy_solution', 'bug_type', 'failure_symptoms', 'entry_point', 'import', 'test_setup', 'test', 'example_test', 'signature', 'docstring', 'instruction'],
        num_rows: 164
    })
})

In [2]:
# looking at some keys:
def print_sample(eidx,keys_to_print=['prompt','test','canonical_solution']):
    print('='*20,f"problem {eidx}",'='*20)
    for key in keys_to_print:
        print('-'*10,key,'-'*10)
        print(ds['test'][eidx][key])

eidx=10
print_sample(eidx)

==================== problem 10 ====================
---------- prompt ----------
#include<stdio.h>
#include<string>
using namespace std;
bool is_palindrome(string str){
    //Test if given string is a palindrome 
    string s(str.rbegin(),str.rend());
    return s==str;
}
string make_palindrome(string str){
    /*
    Find the shortest palindrome that begins with a supplied string. 
    Algorithm idea is simple: - Find the longest postfix of supplied string that is a palindrome. 
    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.
    >>> make_palindrome("") 
    "" 
    >>> make_palindrome("cat") 
    "catac" 
    >>> make_palindrome("cata") 
    "catac" 
    */

---------- test ----------
#undef NDEBUG
#include<assert.h>
int main(){
    assert (make_palindrome("") == "");
    assert (make_palindrome("x") == "x");
     assert (make_palindrome("xyz") == "xyzyx");
     assert (make_palindrome("xyx") == "xyx") ;
     assert (make_pa

In [15]:
# explore the format of generation file
import json

# generations_file='../outputs/generations_codeqwen15_humanevalsynthesize-cpp.json'
generations_file='../outputs/generations_gpt4o_humanevalsynthesize-cpp.json'
# generations_file='../outputs/lang_chain__generated_responses.json'

# Open and read the JSON file
with open(generations_file, 'r') as file:
    generations = json.load(file)

print(f'found {len(generations)} problems, each has {len(generations[0])} generated solutions')

found 164 problems, each has 20 generated solutions


In [18]:
eidx=0
print_sample(eidx,['prompt'])
print('='*30,'generated code','='*30)
print(generations[eidx][6])

==================== problem 0 ====================
---------- prompt ----------
/*
Check if in given vector of numbers, are any two numbers closer to each other than
given threshold.
>>> has_close_elements({1.0, 2.0, 3.0}, 0.5)
false
>>> has_close_elements({1.0, 2.8, 3.0, 4.0, 5.0, 2.0}, 0.3)
true
*/
#include<stdio.h>
#include<vector>
#include<math.h>
using namespace std;
bool has_close_elements(vector<float> numbers, float threshold){

============================== generated code ==============================
/*
Check if in given vector of numbers, are any two numbers closer to each other than
given threshold.
>>> has_close_elements({1.0, 2.0, 3.0}, 0.5)
false
>>> has_close_elements({1.0, 2.8, 3.0, 4.0, 5.0, 2.0}, 0.3)
true
*/
#include<stdio.h>
#include<vector>
#include<math.h>
using namespace std;
bool has_close_elements(vector<float> numbers, float threshold){

    sort(numbers.begin(), numbers.end());
    for (size_t i = 0; i < numbers.size() - 1; ++i) {
        if (fabs(numbers

In [ ]:
eidx=1
all(x==generations[eidx][0] for x in generations[eidx])

In [7]:
import re

def get_last_function_name(prompt: str) -> str:
    # Regular expression pattern to match function definitions with an opening brace '{'
    pattern = r'\b\w+(?:<[^>]+>)?\s+(\w+)\s*\([^)]*\)\s*{'
    
    # Find all matches in the code
    matches = re.findall(pattern, prompt)
    
    if matches:
        # Return the last match, which is the function name of the last unimplemented function
        return matches[-1]
    else:
        return 'XXXXXXXX'

In [ ]:
eidx=50
prompt = ds['test'][eidx]['prompt']
print(prompt)
get_last_function_name(prompt)

In [ ]:
func_names=[get_last_function_name(ds['test'][eidx]['prompt']) for eidx in range(ds['test'].num_rows)]
invalid_indices = [index for index, element in enumerate(func_names) if element == 'XXXXXXXX']
invalid_indices

## Draft evaluation flow

In [1]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")



In [9]:
from openai import OpenAI

SYSTEM_TEMPLATE = """You are a helpful coding assistant.
You are helping a programmer write a C++ function. Write the body of the function and put it in a markdown code block.
Do not write any other code or explanations.
"""

PROMPT_TEMPLATE = """Complete the C++ function {function_name}. Only write the body of the function {function_name}.

```cpp
{prompt}
```
"""
_set_env("OPENAI_API_KEY")
model = OpenAI()

In [ ]:
eidx=1
prompt = ds['test'][eidx]['prompt']
function_name=get_last_function_name(prompt)
prompt_text = PROMPT_TEMPLATE.format(prompt=prompt, function_name=function_name)
completion = model.chat.completions.create(model="gpt-4o",
                                           messages=[
                                                {"role": "system", "content": SYSTEM_TEMPLATE},
                                                {"role": "user", "content": prompt_text}
                                            ],
                                            max_tokens=4096,
                                            temperature=0.2,
                                            top_p=0.95,
                                            stream=False,
                                            n=1
                                            )
response = completion.choices[0].message.content
print(response)

In [20]:
def _postprocess(arg_prompt: str, arg_output: str) -> str:
    """ Postprocess the output. """
    # remove leading ```, ```cpp, and trailing ```
    arg_output = arg_output.strip().lstrip("```cpp").lstrip("```").rstrip("```")

    # remove prompt if it included it
    if arg_output.startswith(arg_prompt):
        arg_output = arg_output[len(arg_prompt):]

    return arg_output


In [ ]:
# print(completion['content'])

print(_postprocess(prompt,response))